In [ ]:
# # Keles et al. (2024) from dandiset 000623: HFB power of microwire LFP and macro iEEG
#
# Paper: Keles et al. (2024), Scientific Data 11, 214, doi:10.1038/s41597-024-03029-1.
# Dataset: DANDI:000623, version 0.240130.2108, the version the paper cites (reference 54); the
# three published versions hold the same 29 assets (`reproduce_keles_2024.ipynb` shows this).
# Claims file: `claims/keles-2024.yaml`, from the version of record and the
# paper's code (github.com/rutishauserlab/bmovie-release-NWB-BIDS at commit 8c35290).
#
# This notebook computes high-frequency broadband (HFB) power as the paper's `prep_filterLFP.py`
# does, for every session and electrode type, and tests the memory and event selectivity of
# channels; it also tests the example units and channels of Figs. 6 and 7 and the comparison of
# modalities, which needs the single-cell result again.
#
# | Claim | Here |
# |---|---|
# | memory-lfp, memory-ieeg | tested |
# | event-lfp, event-ieeg | tested, with the code's scene-cut annotations |
# | fig6-examples, fig7-examples | tested |
# | memory-modalities-differ | tested |
# | nwb-contents | tested, with the clock check of the LFP |
#
# The other claims are in `reproduce_keles_2024.ipynb`.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

import bmovie as bm

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

assets = bm.asset_urls()
with ProcessPoolExecutor(6) as ex:
    S = list(ex.map(bm.load_session, [u for _, u, _ in assets]))
cuts = bm.scenecuts()
ids = [s["identifier"] for s in S]

t_start = time.time()
with ProcessPoolExecutor(4) as ex:
    H = list(ex.map(bm.hfb_job, [(u, bm.hfb_onsets(s, cuts)) for (_, u, _), s in zip(assets, S)]))
print("HFB computed for %d sessions in %.1f min" % (len(H), (time.time() - t_start) / 60))

In [ ]:
rows = []
for s, h in zip(S, H):
    for et in ["LFP_micro", "LFP_macro"]:
        o = h[et]
        if o is None:
            rows.append(dict(session=s["identifier"], type=et, channels=0))
            continue
        ar = pd.Series([n[1:-1] for n in o["names"]]).value_counts().to_dict()
        rows.append(dict(session=s["identifier"], type=et, channels=len(o["names"]),
                         channels_in_file=s["lfp"][et]["shape"][1], **ar,
                         memory_nan=int(np.isnan(o["memory"]).sum()), events_nan=int(np.isnan(o["events"]).sum()),
                         memory_range=(round(float(np.nanmin(o["memory"])), 2), round(float(np.nanmax(o["memory"])), 2)),
                         events_range=(round(float(np.nanmin(o["events"])), 2), round(float(np.nanmax(o["events"])), 2))))
chan = pd.DataFrame(rows).fillna(0)
print(chan.to_string())
print("\nchannels per type and area (all sessions):")
print(chan.groupby("type")[["ACC", "AMY", "HIP", "OFC", "SMA"]].sum().astype(int).to_string())
print("\nHFB is z-scored power (no unit); window means range from %.2f to %.2f"
      % (min(np.nanmin(h[e]["events"]) for h in H for e in h), max(np.nanmax(h[e]["events"]) for h in H for e in h)))

Intermediate steps of the HFB pipeline. For each session, electrode type and task block,
`hfb_session` keeps the median over channels of the power spectrum before the notch filter and
after notch, high-pass and re-referencing, and the mean, s.d. and maximum of each channel's HFB
trace. The next cell prints the line-noise power at 60, 120 and 180 Hz relative to the
neighbouring 5 Hz on either side, before and after the filters, and the distribution of the
per-channel HFB statistics (z-scored per frequency, then averaged over the 11 frequencies, so the
mean is 0 and the s.d. below 1).

In [ ]:
f = H[0]["LFP_micro"]["psd_freqs"]
def line_ratio(psd, f0):
    i0 = np.argmin(np.abs(f - f0))
    nb = (np.abs(f - (f0 - 5)) < 1.0) | (np.abs(f - (f0 + 5)) < 1.0)
    return psd[i0] / psd[nb].mean()
rows = []
for s, h in zip(S, H):
    for et in ["LFP_micro", "LFP_macro"]:
        o = h[et]
        if o is None:
            continue
        for task in ["enc", "recog"]:
            r = dict(session=s["identifier"], type=et, task=task)
            for f0 in [60, 120, 180]:
                r[f"{f0}Hz_before"] = line_ratio(o[f"psd_{task}_before"], f0)
                r[f"{f0}Hz_after"] = line_ratio(o[f"psd_{task}_after"], f0)
            st = o[f"hfbstats_{task}"]
            r.update(hfb_mean_absmax=np.abs(st[:, 0]).max(), hfb_sd_median=np.median(st[:, 1]),
                     hfb_sd_min=st[:, 1].min(), hfb_sd_max=st[:, 1].max(), hfb_max_median=np.median(st[:, 2]))
            rows.append(r)
inter = pd.DataFrame(rows)
print("line-noise power over the neighbouring +-5 Hz (median over channels), by session, type and block:")
print(inter.round(3).to_string(index=False))
print("\nmedian over blocks:")
print(inter.groupby("type")[[c for c in inter.columns if c.endswith("before") or c.endswith("after")]].median().round(3).to_string())
print("\nlargest |channel mean HFB|: %.2g; channel s.d. from %.3f to %.3f (median %.3f)"
      % (inter.hfb_mean_absmax.max(), inter.hfb_sd_min.min(), inter.hfb_sd_max.max(), inter.hfb_sd_median.median()))

The LFP clock. The files give each LFP series a starting time near 0 s, and the Usage Notes
say to "start at 10 seconds before the starting time" when making the LFP timepoints, so the
first sample is 10 s before the movie; the code subtracts the same 10 s. The cell below reads
the HFB with that offset and without it (the stored starting time taken as the time of the first
sample, as the NWB format defines it), in two ways: the channels with an event-selective response
to scene cuts (bootstrap p < 0.05, either direction), and the channels whose HFB in 0.2 to 1.7 s
after a recognition image differs from the 1 s before it (Wilcoxon signed-rank over trials,
p < 0.01). An image onset evokes a response only at the right time, so the second check can tell
the two readings apart where the first, on a continuous movie, may not. The cell after it bounds
the offset more finely.

In [ ]:
from scipy import stats

def chan_table(et, key, sessions=None):
    blocks, rows = [], []
    for s, h in zip(S, H):
        o = h[et]
        if o is None or (sessions is not None and s["identifier"] not in sessions):
            continue
        if key.startswith("memory"):
            _, a, b = bm.memory_design(s)
        else:
            a = (cuts.kind == "change").to_numpy()
            b = (cuts.kind == "cont").to_numpy()
        blocks.append((o[key], a, b))
        rows.append(pd.DataFrame({"session": s["identifier"], "channel": o["names"],
                                  "area_code": [n[1:-1] for n in o["names"]]}))
    t = pd.concat(rows, ignore_index=True)
    t["area"] = t.area_code.map(bm.CHANNEL_AREAS).fillna(t.area_code)
    return blocks, t

for et in ["LFP_micro", "LFP_macro"]:
    for key in ["events", "events_fileclock"]:
        bl, tb = chan_table(et, key)
        p = np.concatenate([bm.observed_selection(V, a, b, False)[1] for V, a, b in bl])
        print(f"{et:10s} {key:17s} channels {len(p)}: event selective (p < 0.05) {int((p < 0.05).sum())} ({100 * (p < 0.05).mean():.1f}%)")
    for clock in ["", "_fileclock"]:
        n_sig = n_tot = 0
        for h in H:
            o = h[et]
            post, pre = o["memory" + clock], o["memory_pre" + clock]
            for c in range(post.shape[0]):
                ok = np.isfinite(post[c]) & np.isfinite(pre[c])
                n_tot += 1
                if ok.sum() >= 10 and stats.wilcoxon(post[c, ok], pre[c, ok]).pvalue < 0.01:
                    n_sig += 1
        name = "10-s offset (Usage Notes, code)" if not clock else "no offset"
        print(f"{et:10s} image onset response, {name}: {n_sig} of {n_tot} channels ({100 * n_sig / n_tot:.1f}%)")

In [ ]:
# the offset more finely: the image-onset count with onsets shifted around the 10-s offset, and
# the time course of the HFB response next to that of the spikes, which are on the trials' clock
def n_responsive(et, post_key, pre_key):
    n_sig = n_tot = 0
    for h in H:
        o = h[et]
        post, pre = o[post_key], o[pre_key]
        for c in range(post.shape[0]):
            ok = np.isfinite(post[c]) & np.isfinite(pre[c])
            n_tot += 1
            if ok.sum() >= 10 and stats.wilcoxon(post[c, ok], pre[c, ok]).pvalue < 0.01:
                n_sig += 1
    return n_sig, n_tot

for et in ["LFP_micro", "LFP_macro"]:
    line = []
    for dd in sorted(bm.CLOCK_SHIFTS + [0.0]):
        key = "" if dd == 0 else f"_d{dd:+.2f}"
        a, b = n_responsive(et, "memory" + key, "memory_pre" + key)
        line.append(f"{10 + dd:.2f} s: {a} ({100 * a / b:.1f}%)")
    print(f"{et}, responsive channels by offset:", "; ".join(line))

edges = np.arange(bm.TC_WINDOW[0], bm.TC_WINDOW[1] + bm.TC_WINDOW[2] / 2, bm.TC_WINDOW[2])
t_bins = edges[:-1]

def latency(curve):
    """Start of the first bin after onset at or above half the peak (0 to 1.5 s), and the
    largest pre-onset value as a fraction of the peak."""
    post = t_bins >= 0
    pk = curve[post].max()
    first = t_bins[post][np.argmax(curve[post] >= pk / 2)]
    return first, curve[~post].max() / pk

def tc_population(curves):
    """Per unit or channel: baseline-subtracted mean over trials, signed so that the response
    0.2 to 1.7 s after onset is positive, and scaled by its own peak; then averaged."""
    out = []
    for m in curves:
        m = m - np.nanmean(m[t_bins < 0])
        sgn = np.sign(np.nanmean(m[(t_bins >= 0.2) & (t_bins < 1.5)]))
        if sgn == 0 or not np.isfinite(sgn):
            continue
        out.append(sgn * m)
    return np.nanmean(out, 0), len(out)

for et in ["LFP_micro", "LFP_macro"]:
    curves = []
    for h in H:
        o = h[et]
        post, pre = o["memory"], o["memory_pre"]
        for c in range(post.shape[0]):
            ok = np.isfinite(post[c]) & np.isfinite(pre[c])
            if ok.sum() >= 10 and stats.wilcoxon(post[c, ok], pre[c, ok]).pvalue < 0.01:
                curves.append(np.nanmean(o["memory_tc"][c], 0))
    pop, n = tc_population(curves)
    lat, pre_frac = latency(pop)
    print(f"{et}: {n} responsive channels; half-peak latency {lat * 1000:.0f} ms after image onset;"
          f" largest value before onset {100 * pre_frac:.0f}% of the peak")
    print("   ", " ".join(f"{v:+.2f}" for v in pop / pop.max()))
curves = []
for s in S:
    ons = bm.recog_trials(s).start_time.to_numpy()
    for st, inU in zip(s["units"].spike_times, bm.units_U(s).in_U):
        if not inU:
            continue
        st = np.asarray(st)
        post = bm.window_counts(st, ons, (0.2, 1.7))
        pre = bm.window_counts(st, ons, (-1.5, -0.0001))
        if np.any(post != pre) and stats.wilcoxon(post, pre).pvalue < 0.01:
            cnt = np.array([np.histogram(st - o, bins=edges)[0] for o in ons], float)
            curves.append(cnt.mean(0))
pop, n = tc_population(curves)
lat, pre_frac = latency(pop)
print(f"units: {n} responsive units; half-peak latency {lat * 1000:.0f} ms after image onset;"
      f" largest value before onset {100 * pre_frac:.0f}% of the peak")
print("   ", " ".join(f"{v:+.2f}" for v in pop / pop.max()))
print("bin starts (s):", " ".join(f"{v:+.2f}" for v in t_bins))

## nwb-contents

> Each NWB file holds spike times of all sorted neurons, LFP from all microwires and field
> potentials from all macro contacts downsampled to 1000 Hz, behavior, electrode locations,
> spike-sorting quality metrics and eye-tracking data; spike times are referenced to movie
> start and the LFP starts 10 s before it.

In [ ]:
need_units = {"spike_times", "electrode_id", "isibelow", "peakSNR", "meanSNR", "isolationdist"}
chk = []
for s in S:
    t0, _ = bm.movie_window(s)
    lf = s["lfp"]
    chk.append(dict(
        session=s["identifier"],
        units=need_units <= set(s["units_columns"]) and len(s["units"]) > 0,
        micro_1kHz=("LFP_micro" in lf and lf["LFP_micro"]["rate"] == 1000.0),
        macro_1kHz=("LFP_macro" in lf and lf["LFP_macro"]["rate"] == 1000.0),
        lfp_start_file=round(min(v["starting_time"] for v in lf.values()), 4),
        lfp_end_after_last_trial=round(min(v["starting_time"] - bm.LFP_PAD_S + v["shape"][0] / v["rate"]
                                           for v in lf.values()) - float(s["trials"].stop_time.max()), 1),
        trials=len(s["trials"]) > 1, locations="location" in s["electrodes"].columns,
        pairwise="pairwise_distances" in s["electrodes"].columns,
        gaze="EyeTracking" in s["behavior_modules"], frames=len(s["frame_times"]) == 11971,
        movie_start=t0, movie_start_ok=abs(t0) <= 0.04))
chk = pd.DataFrame(chk).set_index("session")
print(chk.to_string())
cols = ["units", "micro_1kHz", "macro_1kHz", "trials", "locations", "pairwise", "gaze", "frames", "movie_start_ok"]
print("\nall files pass every listed item:", bool(chk[cols].all().all()))
print("LFP series whose stored starting_time is the movie start (within 0.01 s), so that by the Usage Notes",
      "the first sample is 10 s before the movie:",
      int(sum(abs(v["starting_time"] - bm.movie_window(s)[0]) < 0.01 for s in S for v in s["lfp"].values())),
      "of", sum(len(s["lfp"]) for s in S))
print("LFP series that, read that way, run past the last trial's stop time:",
      int((chk.lfp_end_after_last_trial >= 0).sum()), "of", len(chk), "files (seconds to spare, smallest per file:",
      chk.lfp_end_after_last_trial.min(), "to", chk.lfp_end_after_last_trial.max(), ")")

**Result: reproduced.** Every file has units with spike times, electrode ids and the four quality
metrics, LFP_micro and LFP_macro at 1000 Hz, the trials table, electrode locations, pairwise
distances, gaze and 11971 frame times, and the movie starts at 0 s in all 29. All 58 LFP series
store a starting time at the movie start, which, read as the Usage Notes say ("start at 10
seconds before the starting time"), puts the first sample 10 s before the movie, and every file's
LFP then runs past the last trial (by 1.0 to 11.0 s). The image-onset check supports that reading:
with the 10-s offset 650 of 2192 micro and 683 of 2240 macro channels respond to recognition
images (30%), without it 18 and 20 (under 1%). Finer, the count stays high for offsets of 9.9 to
10.25 s, and with the 10-s offset the HFB response rises at 200 ms (macro) and 350 ms (micro)
after the image, against 300 ms for the spikes, which are on the trials' clock; the largest value
before onset is 9% of the peak (micro), 7% (macro) and 10% (units). These checks do not pin the
offset to a single bin: the counts keep rising up to 10.25 s (micro 650, 693, 695 and macro 683,
715, 724 at 10.0, 10.1 and 10.25 s), and shifting the onsets also moves the analysis window
against the response, so the sweep bounds the offset only to roughly 9.9 to 10.25 s; the macro
HFB rising 100 ms before the spikes is consistent with a residual offset of about 100 ms but
does not show one below it. The claim needs only 10 s against 0 s, which the checks settle. The stored starting time is
not the time of the first sample, as the NWB format defines `starting_time`, so a reader who
does not apply the Usage Notes' offset reads the LFP 10 s off.

## memory-lfp and memory-ieeg

> For microwire LFP HFB power, significant proportions of channels were memory selective in the
> amygdala (37/368, p = 0.002), ACC (33/368, p = 0.028) and preSMA (40/368, p = 0.001), but not
> in the hippocampus (17/368, p = 0.90) or vmPFC (28/336, p = 0.05).

> For macroelectrode iEEG HFB power, significant proportions of channels were memory selective
> in the ACC (41/368, p < 0.001), vmPFC (34/368, p = 0.012) and preSMA (45/368, p < 0.001), but
> not in the amygdala (25/368, p = 0.47) or hippocampus (32/368, p = 0.067).

Sessions by rule R (the five sessions of reading (a) of memory-sessions-included; the readings
remove the same five sessions, `reproduce_keles_2024.ipynb`). Mean HFB in 0.2 to 1.7 s after
each trial's start_time; correct trials; bootstrap B (novel against familiar, balanced, seed
13); permutation test P with 2,000 shuffles.

In [ ]:
mem_sessions = {s["identifier"] for s in S if bm.rule_R(s)[0]}
print("sessions in the memory analyses:", len(mem_sessions), "; removed:", sorted(set(ids) - mem_sessions))
paper_mem = {
    "LFP_micro": pd.DataFrame({"area": bm.AREAS, "paper_selective": [37, 17, 33, 40, 28],
                               "paper_n": [368, 368, 368, 368, 336], "paper_p": [0.002, 0.90, 0.028, 0.001, 0.05]}),
    "LFP_macro": pd.DataFrame({"area": bm.AREAS, "paper_selective": [25, 32, 41, 45, 34],
                               "paper_n": [368] * 5, "paper_p": [0.47, 0.067, 0.001, 0.001, 0.012]}),
}
res_mem = {}
for k, et in enumerate(["LFP_micro", "LFP_macro"]):
    bl, tb = chan_table(et, "memory", mem_sessions)
    res_mem[et] = bm.selection_and_null(bl, tb, balanced=True, positive_only=False, seed=0)

In [ ]:
t, sm = res_mem["LFP_micro"]
print(sm.merge(paper_mem["LFP_micro"]).round(4).to_string())
print("channels outside the five areas:", int((~t.area.isin(bm.AREAS)).sum()))
print("selective channels with novel > familiar:", int((t.selective & (t["diff"] > 0)).sum()), "of", int(t.selective.sum()))
print("by participant:")
print(t.assign(p=t.session.str[:5]).groupby("p").agg(sel=("selective", "sum"), n=("selective", "size")).T.to_string())

**Result: reproduced.** Amygdala 37/368 (permutation p = 0.0015), ACC 32/368 (p = 0.0315) and
preSMA 40/368 (p = 0.0005) are below 0.05; hippocampus 17/368 (p = 0.896) and vmPFC 28/336
(p = 0.0645, Monte Carlo s.e. 0.0055) are not. Every count is within tolerance (four are exact)
and the denominators are the paper's.

In [ ]:
t, sm = res_mem["LFP_macro"]
print(sm.merge(paper_mem["LFP_macro"]).round(4).to_string())
print("channels outside the five areas:", int((~t.area.isin(bm.AREAS)).sum()))
print("selective channels with novel > familiar:", int((t.selective & (t["diff"] > 0)).sum()), "of", int(t.selective.sum()))
print("by participant:")
print(t.assign(p=t.session.str[:5]).groupby("p").agg(sel=("selective", "sum"), n=("selective", "size")).T.to_string())

**Result: reproduced.** ACC 41/368 (p = 0.0005), vmPFC 34/368 (p = 0.0155) and preSMA 45/368
(p = 0.0005) are below 0.05; amygdala 25/368 (p = 0.468) and hippocampus 31/368 (p = 0.0935) are
not. The counts are the paper's except hippocampus (31 against 32), within tolerance.

## event-lfp and event-ieeg

> For microwire LFP HFB power, significant proportions of channels were event selective in all
> five areas; amygdala 20/448 (p = 0.005), hippocampus 40/448 (p < 0.001), ACC 33/448
> (p < 0.001), vmPFC 15/400 (p = 0.015), preSMA 22/448 (p = 0.001).

> For macroelectrode iEEG HFB power, significant proportions of channels were event selective in
> all five areas; amygdala 67/448, hippocampus 41/448, ACC 25/448, vmPFC 33/449 (all
> p < 0.001) and preSMA 21/448 (p = 0.002).

All 29 sessions; mean HFB in 0 to 1.0 s after each cut onset; bootstrap B (scene change against
continuity cut, each group its own size, seed 13), counted when scene change is above
continuity cut; permutation test P with the same rule.

In [ ]:
paper_ev = {
    "LFP_micro": pd.DataFrame({"area": bm.AREAS, "paper_selective": [20, 40, 33, 22, 15],
                               "paper_n": [448, 448, 448, 448, 400], "paper_p": [0.005, 0.001, 0.001, 0.001, 0.015]}),
    "LFP_macro": pd.DataFrame({"area": bm.AREAS, "paper_selective": [67, 41, 25, 21, 33],
                               "paper_n": [448, 448, 448, 448, 449], "paper_p": [0.001, 0.001, 0.001, 0.002, 0.001]}),
}
res_ev = {}
for k, et in enumerate(["LFP_micro", "LFP_macro"]):
    bl, tb = chan_table(et, "events")
    res_ev[et] = bm.selection_and_null(bl, tb, balanced=False, positive_only=True, seed=0)

In [ ]:
t, sm = res_ev["LFP_micro"]
print(sm.merge(paper_ev["LFP_micro"]).round(4).to_string())
print("variant, either direction:", {a: int(t.selective_any_direction[t.area == a].sum()) for a in bm.AREAS})
print("by participant:")
print(t.assign(p=t.session.str[:5]).groupby("p").agg(sel=("selective", "sum"), n=("selective", "size")).T.to_string())

**Result: reproduced.** All five areas are below 0.05 with the paper's counts: amygdala 20/448
(p = 0.0035), hippocampus 40/448 (p = 0.0005), ACC 33/448 (p = 0.0005), vmPFC 15/400
(p = 0.0175) and preSMA 22/448 (p = 0.0020). In either direction 46 to 77 channels per area are
selective.

In [ ]:
t, sm = res_ev["LFP_macro"]
print(sm.merge(paper_ev["LFP_macro"]).round(4).to_string())
print("variant, either direction:", {a: int(t.selective_any_direction[t.area == a].sum()) for a in bm.AREAS})
print("by participant:")
print(t.assign(p=t.session.str[:5]).groupby("p").agg(sel=("selective", "sum"), n=("selective", "size")).T.to_string())

**Result: reproduced.** All five areas are below 0.05: amygdala 66/448, hippocampus 41/448, ACC
25/448, vmPFC 33/448 and preSMA 21/448 (each p = 0.0005). The counts are the paper's but amygdala
(66 against 67), within tolerance. The vmPFC denominator is 448, so the paper's 449 is a
misprint.

## memory-modalities-differ

> The areas with memory selectivity differ by modality; the ACC is selective in all three
> (cells, LFP, iEEG), the hippocampus in none, and the amygdala in cells and LFP but not iEEG.

The single-cell result is computed again as in `reproduce_keles_2024.ipynb` (same seeds, so the
same numbers).

In [ ]:
blocks_m, table_m = bm.cell_blocks(S, "memory", sessions=mem_sessions)
cells_m, sum_m = bm.selection_and_null(blocks_m, table_m, balanced=True, positive_only=False, seed=0,
                                      rate_window=bm.MEMORY_WINDOW[1] - bm.MEMORY_WINDOW[0])
pt = pd.DataFrame({"cells": sum_m.set_index("area").p_perm,
                   "LFP": res_mem["LFP_micro"][1].set_index("area").p_perm,
                   "iEEG": res_mem["LFP_macro"][1].set_index("area").p_perm})
print(pt.round(4).to_string())
print()
print((pt < 0.05).to_string())
sig = pt < 0.05
print("\nACC in all three:", bool(sig.loc["ACC"].all()), "; hippocampus in none:", bool((~sig.loc["hippocampus"]).all()),
      "; amygdala in cells and LFP but not iEEG:", bool(sig.loc["amygdala", "cells"] and sig.loc["amygdala", "LFP"] and not sig.loc["amygdala", "iEEG"]))

**Result: reproduced.** At p < 0.05 the ACC is memory selective in cells (p = 0.0005), LFP
(p = 0.0315) and iEEG (p = 0.0005); the hippocampus in none (0.282, 0.896, 0.0935); and the
amygdala in cells (0.0095) and LFP (0.0015) but not iEEG (0.468).

## fig6-examples

> Example memory-selective units and channels; a preSMA neuron firing more to familiar images
> (p = 0.003), an ACC neuron firing more to novel images (p < 0.001), and a preSMA macro channel
> and an ACC microwire channel with higher HFB to novel images (both p < 0.001).

In [ ]:
def find(t, prefix, col):
    m = t[t[col].str.startswith(prefix)]
    return m
ex = []
for pid in ["P53CS_R1_11_1_3077", "P58CS_R1_34_2_3239"]:
    m = find(cells_m, pid, "unit_id_session")
    for _, r in m.iterrows():
        ex.append(dict(example=r.unit_id_session, kind="cell", area=r.area, diff_novel_minus_familiar=r["diff"], p=r.p))
for et, cid in [("LFP_macro", "P42CS_R2_RSMA3"), ("LFP_micro", "P58CS_R1_RACC2")]:
    t = res_mem[et][0]
    m = t[(t.session + "_" + t.channel) == cid]
    for _, r in m.iterrows():
        ex.append(dict(example=cid, kind=et, area=r.area, diff_novel_minus_familiar=r["diff"], p=r.p))
print(pd.DataFrame(ex).to_string())
print("(cell differences in spikes/s; channel differences in z-scored HFB)")

**Result: reproduced.** The preSMA cell P53CS_R1_11_1_3077 fires more to familiar images
(-3.13 spikes/s, p = 0.0030); the ACC cell P58CS_R1_34_2_3239 fires more to novel images (1.69
spikes/s, p = 0.0002); the preSMA macro channel P42CS_R2_RSMA3 and the ACC micro channel
P58CS_R1_RACC2 have higher HFB to novel images (p = 0.0002 each, the smallest p of 10,000
resamples).

## fig7-examples

> Example event-selective units and channels; a preSMA neuron (p = 0.024) and a vmPFC neuron
> (p = 0.030), a hippocampal macro channel (p = 0.026) and a preSMA microwire channel
> (p < 0.001), each responding differently to scene changes than to continuity cuts.

In [ ]:
blocks_e, table_e = bm.cell_blocks(S, "event", cuts=cuts)
de, pe = [], []
for V, a, b in blocks_e:
    d, p = bm.observed_selection(V, a, b, False)
    de.append(d)
    pe.append(p)
table_e["diff"] = np.concatenate(de)
table_e["p"] = np.concatenate(pe)
ex = []
for pid in ["P42CS_R1_47_2_1347", "P47CS_R1_78_1_2233"]:
    for _, r in find(table_e, pid, "unit_id_session").iterrows():
        ex.append(dict(example=r.unit_id_session, kind="cell", area=r.area, diff_change_minus_cont=r["diff"], p=r.p))
for et, cid in [("LFP_macro", "P62CS_R2_RHIP6"), ("LFP_micro", "P51CS_R2_RSMA8")]:
    t = res_ev[et][0]
    for _, r in t[(t.session + "_" + t.channel) == cid].iterrows():
        ex.append(dict(example=cid, kind=et, area=r.area, diff_change_minus_cont=r["diff"], p=r.p))
print(pd.DataFrame(ex).to_string())
print("(cell differences in spikes/s; channel differences in z-scored HFB)")
print("notebook run time so far: %.1f min" % ((time.time() - t_start) / 60))

**Result: reproduced.** The preSMA cell P42CS_R1_47_2_1347 (p = 0.0244) and the vmPFC cell
P47CS_R1_78_1_2233 (p = 0.0298), the hippocampal macro channel P62CS_R2_RHIP6 (p = 0.0258) and the
preSMA micro channel P51CS_R2_RSMA8 (p = 0.0002) all respond more to scene changes than to
continuity cuts, with the legend's areas and p-values within tolerance.

## Summary

| Claim | Outcome |
|---|---|
| nwb-contents | reproduced; the LFP's stored starting time is the movie start, and the samples begin 10 s before it, as the Usage Notes say |
| memory-lfp | reproduced |
| memory-ieeg | reproduced |
| event-lfp | reproduced |
| event-ieeg | reproduced |
| memory-modalities-differ | reproduced |
| fig6-examples | reproduced |
| fig7-examples | reproduced |

How the analysis differs from the paper's. HFB is computed as `prep_filterLFP.py` does, but
reduced at once to the window means each test uses, with no intermediate files; the line-noise
power and the per-channel HFB statistics are printed as checks of the steps. The permutation
tests are seeded (seed 0, open detail 8 of the claims file) and share each shuffle's bootstrap
resamples among the channels of a session (see `bmovie.py`).

What was found about the dataset. The LFP series store a starting time at the movie start, while
their samples begin 10 s before it. The Usage Notes say so and the paper's code applies the
offset, but the NWB format defines `starting_time` as the time of the first sample, so a reader
of the file alone is 10 s off; the image-onset check shows the offset is needed, and bounds it to
roughly 9.9 to 10.25 s. The vmPFC macro denominator is 448, not the 449 the paper prints. Channel areas come
only from the channel names (characters 2 to 4 of `origchannel_name`), as the code reads them.